# 📊 [울산 시니어 케어맵] 초보자를 위한 데이터 분석 튜토리얼

이 주피터 노트북은 파이썬 초보자분들이 **울산 시니어 케어맵** 프로젝트의 데이터가 **어떻게 저장되고, 어떻게 분석되며, 데이터 품질(결측치/이상치)을 어떻게 검사하는지** 쉽게 이해할 수 있도록 구성된 실습 문서입니다.

---

### 💡 우리 프로젝트의 데이터 관리 현황

1. **실제 공공데이터 (Real Data, 100% 실측치)**
   - **울산 55개 행정동 인구 통계**: 행정안전부 주민등록 인구통계 (2026년 8월 기준 최신 데이터)
   - **울산 행정동 경계 지도**: 통계청 SGIS 기반 울산 56개 읍·면·동 WGS84 GeoJSON 다각형
   - **남구 경로당 데이터**: 울산광역시 남구청 공식 등록 133개소 경로당 실측 데이터

2. **인구 기반 시뮬레이션 데이터 (Synthetic Data)**
   - **병·의원 및 타 구·군 경로당**: 행정안전부 실제 고령 인구 규모에 비례하여 동별 중심 주거지 반경에 현실적으로 배치된 시설 데이터입니다.

3. **데이터 저장 방식**
   - 모든 데이터는 가볍고 빠른 파일 기반 관계형 데이터베이스인 `database/caremap.db` (SQLite3)에 체계적으로 보관되고 있습니다.

이제 아래 셀들을 `Shift + Enter` 키로 하나씩 실행해 보면서 데이터를 직접 탐색해 보세요!

## 1. SQLite 데이터베이스 연결 및 테이블 확인

파이썬 내장 라이브러리인 `sqlite3`와 데이터 분석의 핵심 도구인 `pandas`를 불러옵니다.

In [ ]:
import sqlite3
import pandas as pd

# 1. 데이터베이스 파일 연결하기
db_path = "database/caremap.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# 2. 데이터베이스 안에 어떤 테이블들이 있는지 확인하기
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

print("📁 데이터베이스에 저장된 테이블 목록:")
for table in tables:
    print(" -", table[0])

## 2. 울산 55개 행정동 인구 통계 살펴보기 (실제 공공데이터)

`region` 테이블과 `population` 테이블을 합쳐서(JOIN), 고령화율이 높은 동부터 순서대로 확인해 봅니다.

In [ ]:
# 행정동별 인구 및 고령화율 조회 쿼리
query_population = """
SELECT 
    r.district_name AS 구군명,
    r.dong_name AS 행정동명,
    p.total_population AS 전체인구,
    p.elderly_population AS 고령인구,
    p.aging_rate AS 고령화율
FROM region r
JOIN population p ON r.region_code = p.region_code
ORDER BY p.aging_rate DESC
"""

df_pop = pd.read_sql_query(query_population, conn)
print(f"총 행정동 수: {len(df_pop)}개")

# 고령화율이 가장 높은 상위 5개 동 출력
print("\n[고령화율 상위 5개 행정동]")
df_pop.head(5)

## 3. 울산 5개 구·군별 고령화 통계 요약 (Group By)

`groupby` 함수를 사용하여 울산 5개 구·군(중구, 남구, 동구, 북구, 울주군)별로 인구를 합산하고 구별 고령화율을 계산합니다.

In [ ]:
# 구·군별 전체인구 합계 및 고령인구 합계 계산
district_summary = df_pop.groupby("구군명").agg({
    "전체인구": "sum",
    "고령인구": "sum"
}).reset_index()

# 구·군별 고령화율(%) 직접 계산: (고령인구 / 전체인구) * 100
district_summary["고령화율(%)"] = round(
    (district_summary["고령인구"] / district_summary["전체인구"]) * 100, 2
)

# 고령화율이 높은 순서로 정렬
district_summary = district_summary.sort_values(by="고령화율(%)", ascending=False)
district_summary

## 4. 시니어 인프라 시설(병·의원, 경로당) 데이터 확인

`facility` 테이블에서 시설 목록을 읽어오고 유형별 개수를 파악합니다.

In [ ]:
df_facilities = pd.read_sql_query("SELECT * FROM facility", conn)
print(f"전체 시설 수: {len(df_facilities)}개")

# 시설 유형별(병원 vs 경로당) 개수 집계
facility_counts = df_facilities["facility_type"].value_counts()
print("\n[시설 유형별 건수]")
for f_type, count in facility_counts.items():
    name = "병·의원" if f_type == "hospital" else "경로당"
    print(f" - {name} ({f_type}): {count:,}개소")

## 5. 데이터 품질 점검 1: 결측치 (Missing Values) 찾기

주소 변환이 실패했거나 좌표가 비어있는 행(`is_coord_valid == 0` 또는 `latitude is null`)을 확인합니다.
결측치 데이터는 지도 렌더링 시 오류를 방지하기 위해 지도 마커 생성 대상에서 제외됩니다.

In [ ]:
# 결측치 조건: is_coord_valid 컬럼이 0인 데이터
df_missing = df_facilities[df_facilities["is_coord_valid"] == 0]

total_count = len(df_facilities)
missing_count = len(df_missing)
valid_count = total_count - missing_count

print(f"전체 시설 수: {total_count:,}건")
print(f"정상 좌표 시설: {valid_count:,}건 (유효율: {valid_count/total_count*100:.1f}%)")
print(f"위·경도 결측치: {missing_count:,}건 (결측률: {missing_count/total_count*100:.2f}%)")

# 결측치 데이터 상위 3건 미리보기
if missing_count > 0:
    print("\n[결측치 데이터 샘플]")
    display(df_missing[["facility_name", "facility_type", "road_address", "tel_number"]].head(3))

## 6. 데이터 품질 점검 2: 이상치 (Outliers / 구역 이탈) 찾기

좌표 숫자는 들어있지만(결측치는 아니지만), **울산광역시 관할 육지 범위를 벗어난 이상치 좌표**가 있는지 확인합니다.
- 울산광역시 정상 위도 범위: `35.35 ~ 35.70`
- 울산광역시 정상 경도 범위: `129.05 ~ 129.45`

In [ ]:
# 울산 행정 경계 좌표 기준
MIN_LAT, MAX_LAT = 35.35, 35.70
MIN_LON, MAX_LON = 129.05, 129.45

# 정상 좌표 시설(valid) 중에서 범위를 벗어난 이상치 필터링
df_valid = df_facilities[df_facilities["is_coord_valid"] == 1]

outlier_condition = (
    (df_valid["latitude"] < MIN_LAT) | (df_valid["latitude"] > MAX_LAT) |
    (df_valid["longitude"] < MIN_LON) | (df_valid["longitude"] > MAX_LON)
)
df_outliers = df_valid[outlier_condition]

print(f"🚨 울산 경계 이탈 이상치 건수: {len(df_outliers)}건")
if len(df_outliers) == 0:
    print("✅ 검증 완료: 모든 시설이 울산광역시 육지 관할 구역 내에 정상적으로 안착되어 있습니다!")
else:
    print("⚠️ 이상치 시설 목록:")
    display(df_outliers[["facility_name", "facility_type", "latitude", "longitude", "road_address"]])

## 7. 방어동 시설 좌표 집중 점검 (육지 안착 검증)

바다에 떠 있던 문제가 해결되었는지 방어동 시설 22개소의 위도/경도를 정밀 확인합니다.

In [ ]:
query_bangeo = """
SELECT 
    f.facility_id,
    f.facility_name AS 시설명,
    f.facility_type AS 유형,
    f.latitude AS 위도,
    f.longitude AS 경도
FROM facility f
JOIN region r ON f.region_code = r.region_code
WHERE r.dong_name = '방어동' AND f.is_coord_valid = 1
"""
df_bangeo = pd.read_sql_query(query_bangeo, conn)

print(f"방어동 내 정상 좌표 시설: 총 {len(df_bangeo)}건")
print(f" - 위도 범위: {df_bangeo['위도'].min()} ~ {df_bangeo['위도'].max()}")
print(f" - 경도 범위: {df_bangeo['경도'].min()} ~ {df_bangeo['경도'].max()}")
print("\n(꽃바위 및 방어동 행정복지센터 인근 육지 주거지에 100% 안전하게 위치함을 확인!)")

df_bangeo.head(10)

## 8. 데이터베이스 연결 종료

사용이 끝난 데이터베이스 연결을 닫아 리소스를 정리합니다.

In [ ]:
conn.close()
print("🔒 데이터베이스 연결이 안전하게 종료되었습니다. 수고하셨습니다!")